In [2]:
pip install papermill

Note: you may need to restart the kernel to use updated packages.


In [4]:

# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import papermill as pm
import pandas as pd

import json

print("✅ Imports done")

✅ Imports done


In [5]:

# ============================================================
# CELL 2 — Configure notebook paths and output CSVs
# ============================================================
# Each entry: (notebook_path, expected_csv_output_path, gate_name)
# The notebook must save its results df to the csv path in its final cell

GATES = [
    {
        "name"    : "Gate1_WER",
        "notebook": "/Users/abey/Documents/WER_PER_PRODUCTION/wer_prod.ipynb",
        "csv"     : "/Users/abey/Documents/WER_PER_PRODUCTION/WER_TEST/results.csv",
    },
    {
        "name"    : "Gate2_NISQA",
        "notebook": "/Users/abey/Documents/NISQA_PROD/nisqa_prod.ipynb",
        "csv"     : "/Users/abey/Documents/NISQA_PROD/results.csv",
    },
    {
        "name"    : "Gate2_UTMOS",
        "notebook": "/Users/abey/Documents/UTMOS/utmos_prod.ipynb",
        "csv"     : "/Users/abey/Documents/UTMOS/results.csv",
    },
    {
        "name"    : "Gate3_SpeakerSIM",
        "notebook": "/Users/abey/Documents/speaker_similarity/speaker_sim_prod.ipynb",
        "csv"     : "/Users/abey/Documents/speaker_similarity/results.csv",
    },
    {
        "name"    : "Gate3_SER",
        "notebook": "/Users/abey/Documents/SER/ser_prod.ipynb",
        "csv"     : "/Users/abey/Documents/SER/results.csv",
    },
    {
        "name"    : "Gate3_Pitch",
        "notebook": "/Users/abey/Documents/pitch/pitch_prod.ipynb",
        "csv"     : "/Users/abey/Documents/pitch/results.csv",
    },
    {
        "name"    : "Gate4_Duration",
        "notebook": "/Users/abey/Documents/duration/duration_prod.ipynb",
        "csv"     : "/Users/abey/Documents/duration/results.csv",
    },
    {
        "name"    : "Gate4_VAD",
        "notebook": "/Users/abey/Documents/pause_alignment/vad_prod.ipynb",
        "csv"     : "/Users/abey/Documents/pause_alignment/results.csv",
    },
    {
        "name"    : "Gate5_Mix",
        "notebook": "/Users/abey/Documents/gate5/gate5_prod.ipynb",
        "csv"     : "/Users/abey/Documents/gate5/results.csv",
    },
]

# wrapper output
WRAPPER_OUTPUT_DIR = "/Users/abey/Documents/pipeline_output"
os.makedirs(WRAPPER_OUTPUT_DIR, exist_ok=True)

# Anthropic API key
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "your_api_key_here")

print(f"✅ {len(GATES)} gates configured")
print(f"Output dir: {WRAPPER_OUTPUT_DIR}")


✅ 9 gates configured
Output dir: /Users/abey/Documents/pipeline_output


In [6]:

# ============================================================
# CELL 3 — Run all gate notebooks
# ============================================================
# papermill executes each notebook in place and saves an
# executed copy so you can inspect outputs if something fails

execution_status = {}

for gate in GATES:
    name     = gate["name"]
    nb_path  = gate["notebook"]
    executed = nb_path.replace(".ipynb", "_executed.ipynb")

    print(f"\n{'='*50}")
    print(f"Running: {name}")
    print(f"{'='*50}")

    try:
        pm.execute_notebook(
            nb_path,
            executed,
            kernel_name="python3"   # change to "utmos" for UTMOS gate
        )
        print(f"✅ {name} complete")
        execution_status[name] = "SUCCESS"

    except Exception as e:
        print(f"❌ {name} failed: {e}")
        execution_status[name] = f"FAILED: {e}"

print("\n\n========== EXECUTION SUMMARY ==========")
for name, status in execution_status.items():
    icon = "✅" if status == "SUCCESS" else "❌"
    print(f"{icon} {name}: {status}")




Running: Gate1_WER
❌ Gate1_WER failed: [Errno 2] No such file or directory: '/Users/abey/Documents/WER_PER_PRODUCTION/wer_prod.ipynb'

Running: Gate2_NISQA
❌ Gate2_NISQA failed: [Errno 2] No such file or directory: '/Users/abey/Documents/NISQA_PROD/nisqa_prod.ipynb'

Running: Gate2_UTMOS
❌ Gate2_UTMOS failed: [Errno 2] No such file or directory: '/Users/abey/Documents/UTMOS/utmos_prod.ipynb'

Running: Gate3_SpeakerSIM
❌ Gate3_SpeakerSIM failed: [Errno 2] No such file or directory: '/Users/abey/Documents/speaker_similarity/speaker_sim_prod.ipynb'

Running: Gate3_SER
❌ Gate3_SER failed: [Errno 2] No such file or directory: '/Users/abey/Documents/SER/ser_prod.ipynb'

Running: Gate3_Pitch
❌ Gate3_Pitch failed: [Errno 2] No such file or directory: '/Users/abey/Documents/pitch/pitch_prod.ipynb'

Running: Gate4_Duration
❌ Gate4_Duration failed: [Errno 2] No such file or directory: '/Users/abey/Documents/duration/duration_prod.ipynb'

Running: Gate4_VAD
❌ Gate4_VAD failed: [Errno 2] No such f

In [7]:

# ============================================================
# CELL 4 — Load CSVs and build unified dataframe
# ============================================================
# Each gate CSV has Model and Sample columns as keys
# All gates are joined on (Model, Sample)
# Missing gate results show as NaN — does not block other gates

unified_df = None

for gate in GATES:
    name     = gate["name"]
    csv_path = gate["csv"]

    if not os.path.exists(csv_path):
        print(f"⚠️  {name}: CSV not found at {csv_path} — skipping")
        continue

    gate_df = pd.read_csv(csv_path)

    # prefix all columns except Model and Sample with gate name
    # prevents column name collisions across gates
    rename_map = {
        col: f"{name}__{col}"
        for col in gate_df.columns
        if col not in ["Model", "Sample"]
    }
    gate_df = gate_df.rename(columns=rename_map)

    if unified_df is None:
        unified_df = gate_df
    else:
        unified_df = pd.merge(
            unified_df, gate_df,
            on=["Model", "Sample"],
            how="outer"
        )

    print(f"✅ {name} merged — {len(gate_df)} rows, {len(gate_df.columns)} columns")

if unified_df is None:
    raise ValueError("No gate CSVs loaded — check paths and re-run gates")

# save unified results
unified_path = os.path.join(WRAPPER_OUTPUT_DIR, "unified_results.csv")
unified_df.to_csv(unified_path, index=False)
print(f"\n✅ Unified results saved: {unified_path}")
print(f"   {len(unified_df)} segments × {len(unified_df.columns)} columns")



✅ Gate1_WER merged — 6 rows, 19 columns
✅ Gate2_NISQA merged — 4 rows, 17 columns
✅ Gate2_UTMOS merged — 4 rows, 4 columns
✅ Gate3_SpeakerSIM merged — 2 rows, 5 columns
✅ Gate3_SER merged — 4 rows, 10 columns
✅ Gate3_Pitch merged — 2 rows, 16 columns
⚠️  Gate4_Duration: CSV not found at /Users/abey/Documents/duration/results.csv — skipping
✅ Gate4_VAD merged — 2 rows, 16 columns
⚠️  Gate5_Mix: CSV not found at /Users/abey/Documents/gate5/results.csv — skipping

✅ Unified results saved: /Users/abey/Documents/pipeline_output/unified_results.csv
   16 segments × 75 columns


In [8]:

# ============================================================
# CELL 5 — Build gate-level pass/fail summary per segment
# ============================================================
# Identify the Final Pass column from each gate
# Collapse to one pass/fail per gate per segment

GATE_PASS_COLUMNS = {
    "Gate1_WER"       : "Gate1_WER__Both_Pass",
    "Gate2_NISQA"     : "Gate2_NISQA__Final",
    "Gate2_UTMOS"     : "Gate2_UTMOS__Pass",
    "Gate3_SpeakerSIM": "Gate3_SpeakerSIM__Pass",
    "Gate3_SER"       : "Gate3_SER__Pass",
    "Gate3_Pitch"     : "Gate3_Pitch__Final Pass",
    "Gate4_Duration"  : "Gate4_Duration__Final Pass",
    "Gate4_VAD"       : "Gate4_VAD__Final Pass",
    "Gate5_Mix"       : "Gate5_Mix__Final Pass",
}

def is_pass(val):
    if pd.isna(val):
        return None
    return "✅" in str(val)

for gate_name, col in GATE_PASS_COLUMNS.items():
    if col in unified_df.columns:
        unified_df[f"_pass_{gate_name}"] = unified_df[col].apply(is_pass)
    else:
        print(f"⚠️  Column not found: {col}")
        unified_df[f"_pass_{gate_name}"] = None

# overall pass — segment must pass all gates
pass_cols = [f"_pass_{g}" for g in GATE_PASS_COLUMNS]
unified_df["Overall_Pass"] = unified_df[pass_cols].apply(
    lambda row: all(v is True for v in row if v is not None),
    axis=1
)

unified_df.to_csv(unified_path, index=False)
print("✅ Gate-level pass/fail columns added")



⚠️  Column not found: Gate4_Duration__Final Pass
⚠️  Column not found: Gate5_Mix__Final Pass
✅ Gate-level pass/fail columns added


In [9]:

# ============================================================
# CELL 6 — Model ranking summary
# ============================================================
print("\n========== MODEL RANKING SUMMARY ==========")
summary_rows = []

for model in unified_df["Model"].unique():
    model_df = unified_df[unified_df["Model"] == model]
    total    = len(model_df)

    row = {"Model": model, "Total Segments": total}

    for gate_name in GATE_PASS_COLUMNS:
        col        = f"_pass_{gate_name}"
        pass_count = model_df[col].apply(lambda x: x is True).sum()
        row[f"{gate_name} Pass"] = f"{pass_count}/{total}"

    overall_pass = model_df["Overall_Pass"].sum()
    row["Overall Pass Rate"] = f"{overall_pass}/{total}"

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df["_overall_num"] = summary_df["Overall Pass Rate"].apply(
    lambda x: int(x.split("/")[0])
)
summary_df = summary_df.sort_values("_overall_num", ascending=False).drop(
    columns=["_overall_num"]
)

print(summary_df.to_string(index=False))

summary_path = os.path.join(WRAPPER_OUTPUT_DIR, "model_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\n✅ Model summary saved: {summary_path}")




========== MODEL RANKING SUMMARY ==========
  Model  Total Segments Gate1_WER Pass Gate2_NISQA Pass Gate2_UTMOS Pass Gate3_SpeakerSIM Pass Gate3_SER Pass Gate3_Pitch Pass Gate4_Duration Pass Gate4_VAD Pass Gate5_Mix Pass Overall Pass Rate
     m1               6            0/6              2/6              1/6                   2/6            2/6              2/6                 0/6            0/6            0/6               3/6
     m2               4            0/4              2/4              1/4                   0/4            2/4              0/4                 0/4            0/4            0/4               3/4
model_1               3            3/3              0/3              0/3                   0/3            0/3              0/3                 0/3            0/3            0/3               3/3
model_2               3            3/3              0/3              0/3                   0/3            0/3              0/3                 0/3            0/3            0/

In [ ]:

# ============================================================
# CELL 7 — LLM interpretation layer
# ============================================================
# Sends each failing segment to Claude with all metric values
# Claude returns actionable fix per segment
# Results saved to llm_feedback.csv

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are a post-production audio quality analyst for Hindi-to-English dubbing.
You receive metric results for a single TTS audio segment evaluated against a reference Hindi recording.
Your job is to identify the root cause of failures and provide one specific, actionable fix for the dubbing editor.
Be concise. One sentence per issue maximum. Use technical audio terms the editor will understand.
Respond only in JSON with this structure:
{
  "root_cause": "brief description of primary issue",
  "fixes": ["fix 1", "fix 2"],
  "priority": "HIGH | MEDIUM | LOW"
}"""

def build_segment_prompt(row):
    """Build a focused prompt from a segment row — only include relevant failing metrics."""
    lines = [
        f"Model: {row['Model']}",
        f"Sample: {row['Sample']}",
        "",
        "FAILING METRICS:",
    ]

    for gate_name, pass_col in GATE_PASS_COLUMNS.items():
        pass_key = f"_pass_{gate_name}"
        if row.get(pass_key) is False:
            # include the raw metric values for this gate
            gate_cols = [c for c in row.index if c.startswith(f"{gate_name}__") and not c.startswith("_")]
            for col in gate_cols:
                val = row.get(col)
                if pd.notna(val):
                    clean_col = col.replace(f"{gate_name}__", "")
                    lines.append(f"  {gate_name} | {clean_col}: {val}")

    return "\n".join(lines)

llm_results = []
failing_segments = unified_df[unified_df["Overall_Pass"] == False]

print(f"\nSending {len(failing_segments)} failing segments to Claude...")

for _, row in failing_segments.iterrows():
    prompt = build_segment_prompt(row)

    try:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1000,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}]
        )

        raw_text = response.content[0].text.strip()

        # strip markdown fences if present
        if raw_text.startswith("```"):
            raw_text = raw_text.split("```")[1]
            if raw_text.startswith("json"):
                raw_text = raw_text[4:]

        feedback = json.loads(raw_text)

        llm_results.append({
            "Model"      : row["Model"],
            "Sample"     : row["Sample"],
            "Root Cause" : feedback.get("root_cause", "—"),
            "Fixes"      : " | ".join(feedback.get("fixes", [])),
            "Priority"   : feedback.get("priority", "—"),
        })

        print(f"  ✅ {row['Model']} / {row['Sample']} → {feedback.get('priority')} | {feedback.get('root_cause')}")

    except Exception as e:
        print(f"  ❌ {row['Model']} / {row['Sample']} → LLM error: {e}")
        llm_results.append({
            "Model"      : row["Model"],
            "Sample"     : row["Sample"],
            "Root Cause" : "LLM ERROR",
            "Fixes"      : str(e),
            "Priority"   : "—",
        })

llm_df = pd.DataFrame(llm_results)
llm_path = os.path.join(WRAPPER_OUTPUT_DIR, "llm_feedback.csv")
llm_df.to_csv(llm_path, index=False)

print(f"\n✅ LLM feedback saved: {llm_path}")
print(llm_df.to_string(index=False))
